# PyProgressive — Progressive Visualization (Phase 1)

This notebook demonstrates `pp.viz`, which binds PyProgressive's progressive computation to interactive Plotly charts.

Two chart types are available:
- **`pp.vis.line()`** — each variable becomes a line; x-axis = elapsed time, y-axis = current value
- **`pp.vis.scatter()`** — shows the convergence *trajectory* of two variables as a moving point

In [1]:
%pip install -e ..
%pip install plotly anywidget

Obtaining file:///C:/Users/shoon/Desktop/PyProgressive/PyProgressive
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for pyprogressive (pyproject.toml): started
  Building editable for pyprogressive (pyproject.toml): finished with status 'done'
  Created wheel for pyprogressive: filename=pyprogressive-0.1-0.editable-py3-none-any.whl size=2750 sha256=58d187d9e2339f3ec1d4b30e0363fdc9a3127fac33dcc2bcc70ebeef5e285dbf
  Stored in directory: C:\Users\shoon\AppData\Local\Temp\pip-ephem-wheel-cache-i2yg5m5m\wheels\45\8b\9e\eea62cd1f40


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 1. Line Chart — Basic Example

Compute mean and variance of a small array progressively.
The chart updates in real-time as each data point is processed.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum

pp.reset()  # clear arrays from previous cells

data = pp.array([10, 20, 0, 21, 5, 42, 11, 14, 34, 13])

mean = accum(each(data)) / len(data)
var  = accum((each(data) - mean) ** 2) / len(data)

program = pp.compile(mean, var)

chart = pp.vis.line(
    labels=["Mean", "Variance"],
    title="Progressive Mean & Variance"
)
chart.run(program, interval=0.0001)

---
## 2. Line Chart — California Housing Dataset

Compute covariance and variance for multiple features progressively.
Each statistic is one line; watch them converge as more data is processed.

In [3]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()  # clear arrays from previous cells

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"].tolist())
arrayX2 = pp.array(X["HouseAge"].tolist())
arrayY  = pp.array(y.tolist())

mean1  = accum(each(arrayX1)) / len(arrayX1)
mean2  = accum(each(arrayX2)) / len(arrayX2)
meanY  = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
var2   = accum((each(arrayX2) - mean2) ** 2) / len(arrayX2)

covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)
covX2Y = accum((each(arrayX2) - mean2) * (each(arrayY) - meanY)) / len(arrayX2)

program = pp.compile(covX1Y, covX2Y, var1, var2)

chart = pp.vis.line(
    labels=["Cov(MedInc, Price)", "Cov(HouseAge, Price)", "Var(MedInc)", "Var(HouseAge)"],
    title="California Housing — Progressive Statistics"
)
chart.run(program, interval=0.3)

---
## 3. Scatter Chart — Convergence Trajectory

`pp.vis.scatter()` takes exactly 2 variables.

The **blue line** is the trajectory of `(var0, var1)` over time;  
the **red dot** is the current estimate.  
As more data is processed, the dot stabilises at the true value.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()  # clear arrays from previous cells

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"].tolist())
arrayY  = pp.array(y.tolist())

mean1 = accum(each(arrayX1)) / len(arrayX1)
meanY = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)

program = pp.compile(covX1Y, var1)

chart = pp.vis.scatter(
    x_label="Cov(MedInc, Price)",
    y_label="Var(MedInc)",
    title="Convergence Trajectory — Cov vs Var"
)
chart.run(program, interval=0.3)

---
## 4. subplots() — Single Subplot (Matplotlib-like API)

`pp.vis.subplots()` returns `(fig, ax)`, just like `plt.subplots()`.  
Bind variables with `ax.line()` or `ax.scatter()`, configure axes, then call `fig.run()`.

In [5]:
import pyprogressive as pp
from pyprogressive import each, accum

pp.reset()

data = pp.array([10, 20, 0, 21, 5, 42, 11, 14, 34, 13])

mean = accum(each(data)) / len(data)
var  = accum((each(data) - mean) ** 2) / len(data)

program = pp.compile(mean, var)

fig, ax = pp.vis.subplots()
ax.line(mean, label="Mean")
ax.line(var,  label="Variance")
ax.set_title("Progressive Mean & Variance")
ax.set_xlabel("Elapsed Time (s)")
ax.set_ylabel("Value")
fig.run(program, interval=0.0001)

---
## 5. subplots(1, 2) — Line + Scatter Side by Side

`pp.vis.subplots(1, 2)` returns `(fig, [ax1, ax2])`.  
Left panel: two statistics as lines. Right panel: convergence trajectory scatter.

In [6]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"])
arrayY  = pp.array(y)

mean1 = accum(each(arrayX1)) / len(arrayX1)
meanY = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)

program = pp.compile(covX1Y, var1)

fig, (ax1, ax2) = pp.vis.subplots(1, 2)

ax1.line(covX1Y, label="Cov(MedInc, Price)")
ax1.line(var1,   label="Var(MedInc)")
ax1.set_title("Lines over Time")
ax1.set_xlabel("Elapsed Time (s)")

ax2.scatter(covX1Y, var1)
ax2.set_title("Convergence Trajectory")
ax2.set_xlabel("Cov(MedInc, Price)")
ax2.set_ylabel("Var(MedInc)")

fig.run(program, interval=0.3)

---
## 6. Bar Chart — GroupBy Variable

`ax.bar(var)` accepts a **GroupBy variable** (result is a `dict`).
Each group becomes a bar; the heights update progressively as more data is processed.

In [7]:
import pyprogressive as pp
from pyprogressive import each, accum, group, G

pp.reset()

data = pp.array([
    ('A', 2), ('B', 1), ('B', 4), ('C', 3), ('A', -3),
    ('A', 10), ('A', 8), ('B', 7), ('A', 10), ('A', 0),
])

# group sum and group mean
group_sum  = group(each(data, 0), accum(each(G, 1)))
group_mean = group(each(data, 0), accum(each(G, 1)) / accum(1))

program = pp.compile(group_sum, group_mean)

fig, (ax1, ax2) = pp.vis.subplots(1, 2)

ax1.bar(group_sum, label="Sum")
ax1.set_title("Group Sum")
ax1.set_ylabel("Sum")

ax2.bar(group_mean, label="Mean")
ax2.set_title("Group Mean")
ax2.set_ylabel("Mean")

fig.run(program, interval=0.05)

---
## 7. Bar Chart — Scalar Variables (Grouped)

Calling `ax.bar()` multiple times on the same axes adds grouped bar series.
Each bar shows the **current** estimate of that variable — a real-time snapshot.

In [8]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X['MedInc'])
arrayX2 = pp.array(X['HouseAge'])
arrayX3 = pp.array(X['AveRooms'])
arrayY  = pp.array(y)

mean1 = accum(each(arrayX1)) / len(arrayX1)
mean2 = accum(each(arrayX2)) / len(arrayX2)
mean3 = accum(each(arrayX3)) / len(arrayX3)
meanY = accum(each(arrayY))  / len(arrayY)

meanY_var = accum((each(arrayY) - meanY) ** 2) / len(arrayY)

covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)
covX2Y = accum((each(arrayX2) - mean2) * (each(arrayY) - meanY)) / len(arrayX2)
covX3Y = accum((each(arrayX3) - mean3) * (each(arrayY) - meanY)) / len(arrayX3)

var1 = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
var2 = accum((each(arrayX2) - mean2) ** 2) / len(arrayX2)
var3 = accum((each(arrayX3) - mean3) ** 2) / len(arrayX3)

program = pp.compile(covX1Y, covX2Y, covX3Y, var1, var2, var3, meanY_var)

fig, (ax1, ax2) = pp.vis.subplots(1, 2)

# left: covariances as bars (snapshot)
ax1.bar(covX1Y, label='Cov(MedInc,Price)')
ax1.bar(covX2Y, label='Cov(HouseAge,Price)')
ax1.bar(covX3Y, label='Cov(AveRooms,Price)')
ax1.set_title('Covariances (current estimate)')

# right: covariances convergence over time as lines
ax2.line(covX1Y, label='Cov(MedInc,Price)')
ax2.line(covX2Y, label='Cov(HouseAge,Price)')
ax2.line(covX3Y, label='Cov(AveRooms,Price)')
ax2.set_title('Covariances (convergence over time)')
ax2.set_xlabel('Elapsed Time (s)')

fig.run(program, interval=0.3)

---
## 8. Loan Dataset — Covariance & Correlation

`pp.sqrt()` lets you build derived PyProgressive variables — ideal for **Pearson correlation**:

```python
corr = cov_xy / pp.sqrt(var_x * var_y)
```

This is a regular variable: just pass it to `compile()` and plot with `ax.line()`.

Left panel: covariance of two pairs as they converge.  
Right panel: the corresponding Pearson correlation `cov(X,Y) / sqrt(var(X)·var(Y))`.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["loan_amnt", "installment", "annual_inc"],
    nrows=50000,
)
df = df.dropna()

X1 = pp.array(df["loan_amnt"])    # loan amount ($)
X2 = pp.array(df["installment"])  # monthly installment ($)
X3 = pp.array(df["annual_inc"])   # annual income ($)

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)

var1  = accum((each(X1) - mean1) ** 2) / len(X1)
var2  = accum((each(X2) - mean2) ** 2) / len(X2)
var3  = accum((each(X3) - mean3) ** 2) / len(X3)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)

# Pearson correlation = cov(X,Y) / sqrt(var(X) * var(Y))
corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)

program = pp.compile(cov12, cov13, var1, var2, var3, corr12, corr13)

fig, (ax1, ax2) = pp.vis.subplots(1, 2)

ax1.line(cov12, label="Cov(LoanAmt, Installment)")
ax1.line(cov13, label="Cov(LoanAmt, AnnualInc)")
ax1.set_title("Covariance — Loan Dataset")
ax1.set_xlabel("Elapsed Time (s)")

ax2.line(corr12, label="Corr(LoanAmt, Installment)")
ax2.line(corr13, label="Corr(LoanAmt, AnnualInc)")
ax2.set_title("Pearson Correlation — Loan Dataset")
ax2.set_xlabel("Elapsed Time (s)")
ax2.set_ylabel("Correlation")

fig.run(program, interval=0.3)

---
## 9. Yellow Taxi Trip Data — Covariance & Correlation

NYC taxi trips (100 k rows).  
- **Cov(Distance, Fare)** should be large and positive — longer rides cost more.  
- **Corr(Distance, Fare)** converges toward ~0.9, confirming the strong linear link.  
- **Corr(Distance, Tip)** is weaker, reflecting that tips depend on more than distance.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "yellow_tripdata_2016-01_100k.csv"),
    usecols=["trip_distance", "fare_amount", "tip_amount"],
)
df = df.dropna()
df = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0)]

X1 = pp.array(df["trip_distance"])  # miles
X2 = pp.array(df["fare_amount"])    # dollars
X3 = pp.array(df["tip_amount"])     # dollars

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)

var1  = accum((each(X1) - mean1) ** 2) / len(X1)
var2  = accum((each(X2) - mean2) ** 2) / len(X2)
var3  = accum((each(X3) - mean3) ** 2) / len(X3)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)

# Pearson correlation = cov(X,Y) / sqrt(var(X) * var(Y))
corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)

program = pp.compile(cov12, cov13, var1, var2, var3, corr12, corr13)

fig, (ax1, ax2) = pp.vis.subplots(1, 2)

ax1.line(cov12, label="Cov(Distance, Fare)")
ax1.line(cov13, label="Cov(Distance, Tip)")
ax1.set_title("Covariance — Yellow Taxi Trips")
ax1.set_xlabel("Elapsed Time (s)")

ax2.line(corr12, label="Corr(Distance, Fare)")
ax2.line(corr13, label="Corr(Distance, Tip)")
ax2.set_title("Pearson Correlation — Yellow Taxi Trips")
ax2.set_xlabel("Elapsed Time (s)")
ax2.set_ylabel("Correlation")

fig.run(program, interval=0.3)

---
## 10. Confidence Interval — GroupBy Bar Chart (Auto)

`ax.bar(group_mean, ci=0.95)` automatically computes per-group variance and count
internally — no extra compile needed.

Each bar gets an **error bar** showing the CI half-width.
Groups with fewer samples get wider error bars.

Left panel: no CI. Right panel: same variable with `ci=0.95`.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum, group, G
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["grade", "loan_amnt"],
    nrows=50000,
)
df = df.dropna()

# array of (grade, loan_amnt) tuples — grade is the group key
data = pp.array(list(zip(df["grade"], df["loan_amnt"])))

group_mean = group(each(data, 0), accum(each(G, 1)) / accum(1))

program = pp.compile(group_mean)

fig, (ax1, ax2) = pp.vis.subplots(1, 2, figsize=(1100, 450))

ax1.bar(group_mean, label="Mean Loan Amount")
ax1.set_title("Mean Loan Amount by Grade (no CI)")
ax1.set_ylabel("$ (USD)")

ax2.bar(group_mean, label="Mean Loan Amount", ci=0.95)  # variance & count auto-computed
ax2.set_title("Mean Loan Amount by Grade — 95% CI")
ax2.set_ylabel("$ (USD)")

fig.suptitle("Loan Dataset — Mean Loan Amount by Credit Grade")
fig.run(program, interval=0.1)

---
## 11. Confidence Interval — Line Chart (Mean with CI Band)

`ax.line(mean, ci=0.95, variance=var, n=count)` draws a **fill band** around the line
showing the 95% confidence interval of the mean at each tick.

The band is wide at the start (few samples) and narrows as more data is processed.
`variance=` and `n=` must be compiled alongside the mean variable.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "yellow_tripdata_2016-01_100k.csv"),
    usecols=["fare_amount"],
)
df = df.dropna()
df = df[df["fare_amount"] > 0]

X = pp.array(df["fare_amount"])

mean  = accum(each(X)) / len(X)
var   = accum((each(X) - mean) ** 2) / len(X)
count = accum(1)

program = pp.compile(mean, var, count)

fig, ax = pp.vis.subplots(figsize=(900, 450))
ax.line(mean, label="Mean Fare", ci=0.95, variance=var, n=count)
ax.set_title("Mean Fare Amount — Yellow Taxi (100k rows, 95% CI)")
ax.set_xlabel("Elapsed Time (s)")
ax.set_ylabel("Fare Amount (USD)")
fig.run(program, interval=0.3)

---
## 12. Heatmap — Progressive Correlation Matrix

`ax.heatmap(var_grid)` accepts a 2-D list whose cells are either
PyProgressive variables **or numeric constants** (e.g. `1` on the diagonal).

- `labels=` sets symmetric row/column labels; use `xlabels=`/`ylabels=` for asymmetric ones.
- `zmin=-1, zmax=1` fixes the colour scale (essential for correlation matrices).
- Default colorscale is `"RdBu"`: red = negative, white = 0, blue = positive.

The matrix is a **snapshot** that updates on every tick — watch the off-diagonal
values converge as more data is processed.

In [3]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data

X1 = pp.array(X["MedInc"].tolist())
X2 = pp.array(X["HouseAge"].tolist())
X3 = pp.array(X["AveRooms"].tolist())
X4 = pp.array(X["AveOccup"].tolist())

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)
mean4 = accum(each(X4)) / len(X4)

var1 = accum((each(X1) - mean1) ** 2) / len(X1)
var2 = accum((each(X2) - mean2) ** 2) / len(X2)
var3 = accum((each(X3) - mean3) ** 2) / len(X3)
var4 = accum((each(X4) - mean4) ** 2) / len(X4)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)
cov14 = accum((each(X1) - mean1) * (each(X4) - mean4)) / len(X1)
cov23 = accum((each(X2) - mean2) * (each(X3) - mean3)) / len(X2)
cov24 = accum((each(X2) - mean2) * (each(X4) - mean4)) / len(X2)
cov34 = accum((each(X3) - mean3) * (each(X4) - mean4)) / len(X3)

corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)
corr14 = cov14 / pp.sqrt(var1 * var4)
corr23 = cov23 / pp.sqrt(var2 * var3)
corr24 = cov24 / pp.sqrt(var2 * var4)
corr34 = cov34 / pp.sqrt(var3 * var4)

program = pp.compile(corr12, corr13, corr14, corr23, corr24, corr34)

LABELS = ["MedInc", "HouseAge", "AveRooms", "AveOccup"]

fig, (ax1, ax2) = pp.vis.subplots(1, 2, figsize=(1100, 480))

# Left: heatmap — 4×4 correlation matrix (diagonal = 1)
ax1.heatmap(
    [[1,       corr12, corr13, corr14],
     [corr12,  1,      corr23, corr24],
     [corr13,  corr23, 1,      corr34],
     [corr14,  corr24, corr34, 1     ]],
    labels=LABELS,
    zmin=-1, zmax=1,
)
ax1.set_title("Correlation Matrix (Progressive)")

# Right: line convergence of selected off-diagonal entries
ax2.line(corr12, label="Corr(MedInc, HouseAge)")
ax2.line(corr13, label="Corr(MedInc, AveRooms)")
ax2.line(corr14, label="Corr(MedInc, AveOccup)")
ax2.set_title("Off-diagonal Correlations over Time")
ax2.set_xlabel("Elapsed Time (s)")
ax2.set_ylabel("Pearson r")

fig.suptitle("California Housing — 4×4 Correlation Matrix")
fig.run(program, interval=0.3)

---
## 13. Pie / Donut Chart — Progressive GroupBy Proportions

`ax.pie(group_var)` renders a **GroupBy variable** as a pie chart.
The slice sizes update as a snapshot on every tick, so you can watch
the proportions shift as more data is processed.

- `hole=0.0` (default) → full pie; `hole=0.4` → donut.
- `colors=` accepts a list of CSS/hex color strings.
- High-level shortcut: `pp.vis.pie(hole=0.4).run(program, interval=...)`

Left panel: loan grade distribution (full pie).  
Right panel: same data as a donut, side-by-side with the bar chart from example 10.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum, group, G
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["grade", "loan_amnt"],
    nrows=50000,
)
df = df.dropna()

data = pp.array(list(zip(df["grade"], df["loan_amnt"])))

group_count = group(each(data, 0), accum(1))
group_mean  = group(each(data, 0), accum(each(G, 1)) / accum(1))
group_sum  = group(each(data, 0), accum(each(G, 1)))

program = pp.compile(group_count, group_mean, group_sum)

fig, (ax1, ax2, ax3) = pp.vis.subplots(1, 3, figsize=(1300, 450))

# Left: full pie — count per grade
ax1.pie(group_count)
ax1.set_title("Loan Count by Grade")

# Centre: donut — same data
ax2.pie(group_sum, hole=0.45)
ax2.set_title("Loan sum by Grade")

# Right: bar — mean loan amount per grade (for comparison)
ax3.bar(group_mean)
ax3.set_title("Mean Loan Amount by Grade")
ax3.set_ylabel("$ (USD)")

fig.suptitle("Loan Dataset — Grade Distribution & Mean Amount")
fig.run(program, interval=1.0)

---
## 14. axhline / axvline — Reference Lines

`ax.axhline(y)` draws a horizontal line at a fixed y value across the full subplot width.  
`ax.axvline(x)` draws a vertical line at a fixed x value (elapsed time) across the full subplot height.

Both accept `color=`, `linewidth=`, and `linestyle=`
(Plotly names: `"solid"` / `"dot"` / `"dash"` / `"longdash"` / `"dashdot"`,
or matplotlib shortcuts: `"-"` / `"--"` / `":"` / `"-."`).

Typical use cases:
- `axhline(0)` — zero baseline on a covariance/correlation plot
- `axhline(±1)` — theoretical bounds for correlation
- `axvline(t)` — mark a specific elapsed-time point of interest

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "yellow_tripdata_2016-01_100k.csv"),
    usecols=["trip_distance", "fare_amount", "tip_amount"],
)
df = df.dropna()
df = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0)]

X1 = pp.array(df["trip_distance"])
X2 = pp.array(df["fare_amount"])
X3 = pp.array(df["tip_amount"])

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)

var1 = accum((each(X1) - mean1) ** 2) / len(X1)
var2 = accum((each(X2) - mean2) ** 2) / len(X2)
var3 = accum((each(X3) - mean3) ** 2) / len(X3)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)

corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)

program = pp.compile(cov12, cov13, corr12, corr13)

fig, (ax1, ax2) = pp.vis.subplots(1, 2, figsize=(1100, 450))

# Left: covariance with zero baseline
ax1.line(cov12, label="Cov(Distance, Fare)", color="#2196F3")
ax1.line(cov13, label="Cov(Distance, Tip)",  color="#FF9800")
ax1.axhline(0, color="gray", linestyle="dash", linewidth=1)   # zero baseline
ax1.set_title("Covariance — Yellow Taxi Trips")
ax1.set_xlabel("Elapsed Time (s)")
ax1.set_ylabel("Covariance")

# Right: correlation with ±1 bounds and 0 baseline
ax2.line(corr12, label="Corr(Distance, Fare)", color="#2196F3")
ax2.line(corr13, label="Corr(Distance, Tip)",  color="#FF9800")
ax2.axhline( 0,  color="gray",  linestyle="dash",      linewidth=1)   # zero
ax2.axhline( 1,  color="green", linestyle="longdash",  linewidth=1)   # upper bound
ax2.axhline(-1,  color="green", linestyle="longdash",  linewidth=1)   # lower bound
ax2.axhline( 0.5, color="blue", linestyle="dot",       linewidth=1)   # reference 0.5
ax2.set_title("Pearson Correlation — with Reference Lines")
ax2.set_xlabel("Elapsed Time (s)")
ax2.set_ylabel("Correlation")
ax2.set_ylim(-1.1, 1.1)

fig.suptitle("Yellow Taxi — axhline Demo")
fig.run(program, interval=0.3)